### Concatenate the four original datasets before switching from the cluster to local storage

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load all four original datasets from the cluster

adbc_path = "/home/meganorm-mene/project_mene/P6Datasets/adbc23/adbc23_data/adbc23_erp.csv"
adbc_df = pd.read_csv(adbc_path)

cody_path = "/home/meganorm-mene/project_mene/P6Datasets/cody23/dbc19_data.csv"
cody_df2 = pd.read_csv(cody_path)

imer_path = "/home/meganorm-mene/project_mene/P6Datasets/imer21/CAPExp.csv"
imer_df = pd.read_csv(imer_path)

osf_path = "/home/meganorm-mene/project_mene/P6Datasets/osf_archive/data/dbac25.csv"
osf_df = pd.read_csv(osf_path)

# electrodes common to all four datasets
electrodes = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6', 'T7', 'C3', 'Cz', 'C4', 'T8', 'TP9', 'CP5', 'CP1', 'CP2', 'CP6', 'TP10', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO9', 'O1', 'Oz', 'O2', 'PO10']

In [3]:
datasets = {
    "adbc": adbc_df,
    "cody": cody_df2,
    "imer": imer_df,
    "osf": osf_df
}

all_columns = sorted( set().union(*[df.columns for df in datasets.values()]))

presence = pd.DataFrame({
    name: [col in df.columns for col in all_columns]
    for name, df in datasets.items()
}, index=all_columns)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
print(presence)

                                adbc   cody   imer    osf
ACC                             True  False   True  False
Assoc                          False   True  False  False
Association                    False  False   True   True
Association_MC                 False  False   True  False
Association_RC                 False  False   True  False
Association_weighted           False  False   True  False
C3                              True   True   True   True
C4                              True   True   True   True
CP1                             True   True   True   True
CP2                             True   True   True   True
CP5                             True   True   True   True
CP6                             True   True   True   True
Cloze                           True   True   True   True
ClozeDistractorAB              False  False  False   True
Cloze_Balanced                 False  False   True  False
Cloze_C_alternative             True  False  False  False
Cloze_C_altern

In [4]:
print(len(electrodes))

32


In [5]:
# Prepare datasets for merging
def prepare_dataset(df, site_name):
    df = df.copy() # to avoid breaking something
    df["site"] = site_name # save the site
    df["subject_global"] = site_name + "_" + df["Subject"].astype(str) # create unique global ID as the site name + subject in the original dataset
    # One of the datasets had a different column name for trial -> adjust it
    if "TrialNum" in df.columns:
        df = df.rename(columns={"TrialNum": "Trial"})
    return df # prepared for merging

# prepare all four datasets
adbc_df = prepare_dataset(adbc_df, "adbc")
cody_df2 = prepare_dataset(cody_df2, "cody")
imer_df = prepare_dataset(imer_df, "imer")
osf_df = prepare_dataset(osf_df, "osf")

In [6]:
combined_df = pd.concat(
    [adbc_df, cody_df2, imer_df, osf_df],
    ignore_index=True
) # concatenate datasets

cols = ["site","subject_global","Trial","Timestamp","Cloze", "Condition"] + electrodes # retain only useful columns
combined_original = combined_df[cols]

combined_original.to_csv("original_combined.csv", index = False) # save dataset and transfer from cluster to local memory to continue working

In [10]:
combined_original.head()

,site,subject_global,Trial,Timestamp,Cloze,Condition,Fp1,Fp2,F7,F3,Fz,F4,F8,FC5,FC1,FC2,FC6,T7,C3,Cz,C4,T8,TP9,CP5,CP1,CP2,CP6,TP10,P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10
0,adbc,adbc_1,3,-200,0.6,A,7.52634,6.17037,4.74093,9.71641,14.12198,15.68914,10.77544,6.87662,12.36763,15.65102,13.34560,1.47690,9.55732,10.25377,15.08428,-1.40870,-11.39445,6.81434,9.41013,11.05389,11.83088,-16.72092,3.30937,7.27084,7.44292,9.77244,6.44218,-2.88341,4.29948,2.95338,3.62070,-0.19427
1,adbc,adbc_1,3,-199,0.6,A,8.25349,6.83336,5.54648,10.19336,14.63485,15.91679,11.11336,7.39464,12.95686,16.13424,13.74252,2.83803,10.05462,10.49616,15.63126,-0.42898,-11.08263,7.35831,9.85724,11.27818,12.19093,-17.93675,4.25536,7.78743,7.68300,10.08716,6.70009,-1.57330,4.77531,3.34477,3.93404,0.90741
2,adbc,adbc_1,3,-198,0.6,A,8.75974,7.34799,6.23810,10.50329,15.08764,16.17009,11.28423,7.85290,13.48743,16.60512,14.16441,4.00069,10.50548,10.74980,16.14993,0.20881,-10.90893,7.95096,10.25246,11.55577,12.51372,-18.86252,5.20403,8.29442,7.98370,10.37010,7.04727,-0.63995,5.25787,3.77866,4.32211,1.91255
3,adbc,adbc_1,3,-197,0.6,A,8.96987,7.56042,6.79066,10.65580,15.41176,16.33179,11.21294,8.16334,13.85343,16.97013,14.52894,4.74857,10.85006,11.00170,16.55531,0.45442,-11.03959,8.51164,10.56723,11.85323,12.72364,-19.26993,5.97429,8.77325,8.26969,10.57984,7.38150,-0.28673,5.68636,4.18491,4.72623,2.50286
4,adbc,adbc_1,3,-196,0.6,A,8.99405,7.57924,7.25424,10.74072,15.59703,16.39291,10.96576,8.36769,14.10612,17.24716,14.78084,5.02030,11.11772,11.25754,16.84873,0.47217,-11.39307,9.03556,10.82236,12.19093,12.85581,-19.35677,6.55867,9.21942,8.51199,10.75155,7.62267,-0.39282,6.11809,4.62942,5.17226,2.62132
